# PS S6E6 032 one-vs-rest TabM as-is

Experiment: `exp_20260609_032_ovr_tabm_as_is`

Based on `ps6e6-one-vs-rest-tabm.ipynb`. This notebook keeps the TabM OVR training body and the source notebook's class-wise LogisticRegressionCV over TabM seed outputs. External stacking files are not used. The final submission uses the source-style bias tuning, while the saved OOF/pred `.npy` files are raw normalized probabilities for later blend/correlation checks.

In [1]:
import os
import gc
import json
import random
import warnings
from pathlib import Path
from itertools import combinations
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score, recall_score
from sklearn.linear_model import LogisticRegressionCV
from tqdm import tqdm

try:
    import yaml
except Exception:
    yaml = None

# Optional dependencies used by the source notebook.
# pytabkit is installed in a later cell.


In [2]:
# ============================================================
# 0. Configuration
# ============================================================

EXP_ID = "exp_20260609_032_ovr_tabm_as_is"
COMPETITION = "PS S6E6 Predicting Stellar Class"
CREATED_AT = "2026-06-09"

ID = "id"
TARGET = "class"
NUMS = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
CATS = ["spectral_type", "galaxy_population"]

SEED = 42
N_FOLDS = 5
SEED_LIST = [60, 0, 2809]

# This experiment is ps6e6-one-vs-rest-tabm as-is.
# External submission stacking files are intentionally not used.
ADD_EXT = False
MODEL_TYPES_1 = ["t"]  # QSO
MODEL_TYPES_2 = ["t"]  # STAR
MODEL_TYPES_3 = ["t"]  # GALAXY

PATH = "/kaggle/input/competitions/playground-series-s6e6"
WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR
OUTDIR.mkdir(parents=True, exist_ok=True)

OOF_NPY = OUTDIR / f"oof_{EXP_ID}_proba.npy"
PRED_NPY = OUTDIR / f"pred_{EXP_ID}_proba.npy"
OOF_CSV = OUTDIR / f"oof_{EXP_ID}.csv"
PRED_CSV = OUTDIR / f"pred_{EXP_ID}.csv"
SUBMISSION_PATH = OUTDIR / f"submission_{EXP_ID}.csv"
SUBMISSION_RAW_PATH = OUTDIR / f"submission_raw_{EXP_ID}.csv"
CV_RESULT_PATH = OUTDIR / f"cv_result_{EXP_ID}.json"
FEATURE_INFO_PATH = OUTDIR / f"feature_info_{EXP_ID}.json"
FEATURE_COLUMNS_PATH = OUTDIR / f"feature_columns_{EXP_ID}.csv"
CLASS_DISTRIBUTION_PATH = OUTDIR / f"class_distribution_{EXP_ID}.csv"

CLASS_NAMES = ["GALAXY", "QSO", "STAR"]
target_mapping = {"GALAXY": 0, "QSO": 1, "STAR": 2}
reverse_mapping = {0: "GALAXY", 1: "QSO", 2: "STAR"}

np.random.seed(SEED)
random.seed(SEED)

def metric(y_true, y_pred):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)


In [3]:
# ============================================================
# 1. Load data
# ============================================================

train = pd.read_csv(f"{PATH}/train.csv", index_col=ID)
test = pd.read_csv(f"{PATH}/test.csv", index_col=ID)
sample_submission = pd.read_csv(f"{PATH}/sample_submission.csv")

train[TARGET] = train[TARGET].map(target_mapping)

print("train:", train.shape)
print("test:", test.shape)
display(train.head())


train: (577347, 11)
test: (247435, 10)


,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
id,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,0
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,0
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,1
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,0
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,0


In [4]:
# ============================================================
# 2. Feature engineering from source notebook
# ============================================================

cat_cols = train.select_dtypes(include=['object']).columns.tolist()
num_cols = train.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = [col for col in cat_cols if col not in [ID, TARGET]]
num_cols = [col for col in num_cols if col not in [ID, TARGET]]
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
color_pairs = [
    ('u', 'g'),
    ('u', 'r'),
]
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_g_/_redshift'] = (df['g'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_i_/_redshift'] = (df['i'] / (df['redshift'] + 1e-6)).astype('float32')
    for a, b in color_pairs:
        df[f"_{a}-{b}"] = (df[a] - df[b]).astype('float32')

    # Categorize string cats
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Discretize numericals
    bin_config = {'delta': [100, 500]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')
    return df

comb = feature_engineering(pd.concat([train.drop(columns=TARGET), test], ignore_index=True), fit=True)
y = train[TARGET].copy()
test = comb.iloc[len(y):].reset_index(drop=True)
train = comb.iloc[:len(y)].reset_index(drop=True)
train[TARGET] = y.copy()
del comb 
gc.collect()
new_cat_cols = [col for col in train.columns if col.endswith('_')]
new_num_cols = [col for col in train.columns if col.startswith('_')]

init len(cat_cols): 2
init len(num_cols): 8 



In [5]:
DROP=[c for c in test.columns if test[c].nunique()==1]
print(f"DROP:{DROP}")
train.drop(DROP,axis=1,inplace=True)
test.drop(DROP,axis=1,inplace=True)
new_cat_cols = [col for col in new_cat_cols if col not in DROP]

CATEGORY=CATS+[c for c in test.columns if 'digit' in c]
for c in CATEGORY:

    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))
    

FEATURES=CATEGORY+NUMS
train.head()

DROP:[]


,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,_g_/_redshift,_i_/_redshift,_u-g,_u-r,alpha_cat_,delta_cat_,u_cat_,g_cat_,r_cat_,i_cat_,z_cat_,redshift_cat_,delta_100_quantile_bin_,delta_500_quantile_bin_,class
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0,0,53.536564,47.085331,3.576564,5.114196,0,0,0,0,0,0,0,0,43,215,0
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0,0,120.821922,109.041748,1.691447,3.191300,1,1,1,1,1,1,1,0,66,331,0
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,3,1,7.464886,7.289058,-0.043925,-0.136637,2,2,2,0,2,2,2,1,71,358,1
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0,0,39.266460,34.257919,2.254320,4.287302,3,3,3,0,3,3,3,0,91,457,0
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0,0,35.035980,32.207016,2.231478,3.468709,4,4,2,1,4,1,3,0,46,233,0


In [6]:
TE_columns = []

columns = NUMS + CATS
good_columns = NUMS + CATS

important_combos = [
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
]
important_combos = sorted(important_combos)

for cols in important_combos:
    name = '-'.join(cols)

    train[name] = train[cols[0]].astype(str)
    for col in cols[1:]:
        train[name] = train[name] + '_' + train[col].astype(str)

    test[name] = test[cols[0]].astype(str)
    for col in cols[1:]:
        test[name] = test[name] + '_' + test[col].astype(str)

    combined = pd.concat([train[name], test[name]], ignore_index=True)
    combined, _ = combined.factorize()
    if pd.Series(combined).nunique() > len(combined) // 2:
        train = train.drop(name, axis=1)
        test = test.drop(name, axis=1)
        continue
    train[name] = combined[:len(train)]
    test[name] = combined[len(train):len(train) + len(test)]

    TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

In [7]:
cat_cols = CATS + new_cat_cols

target_mapping = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
reverse_mapping = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
train_full = train.copy()
test_df = test.copy().reset_index(drop=True)
del train, test
gc.collect()
train_full['id'] = train_full.index
train_full[TARGET] = train_full[TARGET].map(reverse_mapping)
submission = pd.read_csv(f'{PATH}/sample_submission.csv')
test_df['id'] = submission['id']

feature_cols = [c for c in train_full.columns if c not in ['id', TARGET] and train_full[c].dtype != 'object']
feature_cols = cat_cols + [c for c in feature_cols if c not in cat_cols]

print(f'Total features: {len(feature_cols)}')
print(f'Engineered features: {[c for c in feature_cols if c not in CATS + NUMS]}')

Total features: 26
Engineered features: ['alpha_cat_', 'delta_cat_', 'u_cat_', 'g_cat_', 'r_cat_', 'i_cat_', 'z_cat_', 'redshift_cat_', 'delta_100_quantile_bin_', 'delta_500_quantile_bin_', '_g_/_redshift', '_i_/_redshift', '_u-g', '_u-r', 'alpha_cat_-delta_cat_', 'u_cat_-z_cat_']


In [8]:
y = train_full[TARGET].map(target_mapping).values

for col in cat_cols:
    combined_cats = pd.concat([train_full[col], test_df[col]]).astype(str).unique()
    train_full[col] = pd.Categorical(train_full[col].astype(str), categories=combined_cats)
    test_df[col] = pd.Categorical(test_df[col].astype(str), categories=combined_cats)

features = train_full[feature_cols].copy()
test_features = test_df[feature_cols].copy()
test_ids = test_df['id'].values

for c in cat_cols:
    features[c] = features[c].cat.codes.astype('int32')
    test_features[c] = test_features[c].cat.codes.astype('int32')

In [9]:
y_0 = (y == 0).astype('int')
y_1 = (y == 1).astype('int')
y_2 = (y == 2).astype('int')
target = pd.DataFrame({'GALAXY': y_0, 'QSO': y_1, 'STAR': y_2})

In [10]:
# ============================================================
# 2. Helpers
# ============================================================

def to_logits(df):
    df_col = df.columns
    epsilon = 1e-7
    preds_x = np.clip(df, epsilon, 1 - epsilon)
    logits = np.log(preds_x / (1 - preds_x))
    preds_x = pd.DataFrame(logits)
    preds_x.columns = df_col
    return preds_x

def public_preds(proba, bias):
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

def tune_bias(proba, y_true):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]

    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_true, public_preds(proba, c))
                    if s > best_score + 1e-9:
                        best_bias, best_score, improved = c, s, True
                        history.append(best_score)
    return best_bias, best_score, history

# No external stacking files in 032 as-is.
add_oof = []
add_test = []
add_features = []
L = 0


In [11]:
# ============================================================
# 3. Install / import TabM
# ============================================================

!pip install -q pytabkit

import torch
import pytabkit
from pytabkit import TabM_D_Classifier

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 57.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [12]:
# ============================================================
# 4. TabM parameters
# ============================================================

params_t = {
    'device': 'cuda',
    'val_metric_name': '1-auc_ovr',
    'random_state': 42,
    'verbosity': 0,
    'tabm_k': 32,
    'num_emb_type': 'pwl',
    'd_embedding': 12,
    'batch_size': 64,
    'lr': 3e-3,
    'weight_decay': 2e-2,
    'n_epochs': 3,
    'dropout': 0.1,
    'd_block': 512,
    'n_blocks': 3
}

In [13]:
# ============================================================
# 5. OVR TabM training functions
# ============================================================

def run(model_type, target_id, drop_id, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = target_id
    target_name = reverse_mapping[target_id]
    for fold, (tr_idx, val_idx) in enumerate(skf.split(features, target[target_name])):
        X_train = features.iloc[tr_idx].reset_index(drop=True)
        X_val = features.iloc[val_idx].reset_index(drop=True)
        X_val_full = features.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        # masking third class
        if drop_id:
            drop_name = reverse_mapping[drop_id]
            train_mask_2 = (train_target[drop_name] == 0)
            val_mask_2 = (val_target[drop_name] == 0)
        else:
            train_mask_2 = (train_target[target_name] >= 0)
            val_mask_2 = (val_target[target_name] >= 0)
        X_train = X_train[train_mask_2]
        train_target = train_target[train_mask_2]
        X_val = X_val[val_mask_2]
        val_target = val_target[val_mask_2]
 
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_features.copy()
    
        encoder = TargetEncoder(cv=5, random_state=seed, smooth='auto')
        result = pd.DataFrame(encoder.fit_transform(X_train[TE_columns], y[tr_idx][train_mask_2]))
        X_train = pd.concat([result.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_val[TE_columns]))
        X_val = pd.concat([result.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_val_full[TE_columns]))
        X_val_full = pd.concat([result.reset_index(drop=True), X_val_full.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_test[TE_columns]))
        X_test = pd.concat([result.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
        X_train = X_train.drop(TE_columns, axis=1)
        X_val = X_val.drop(TE_columns, axis=1)
        X_val_full = X_val_full.drop(TE_columns, axis=1)
        X_test = X_test.drop(TE_columns, axis=1)
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        # print(X_train.shape)
        # print(X_val.shape)
        # print(X_val_full.shape)
        # print(X_test.shape)
        print(val_target.columns[i])
        
        if model_type == 't':
            model = TabM_D_Classifier(**params_t)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i])
        else:
            raise ValueError(f'Unsupported model_type in 032: {model_type}')

        p_val = model.predict_proba(X_val)[:, 1]
        p_val_full = model.predict_proba(X_val_full)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]

        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_1_t = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_1_t.to_parquet(f'oof_preds_{target_id}_{model_type}_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    test_preds_1_t = pd.DataFrame(p_test_x_final, columns=target.columns)
    test_preds_1_t['id'] = test_ids
    test_preds_1_t.to_parquet(f'test_preds_{target_id}_{model_type}_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return h

In [14]:
# Source notebook's class-wise LogisticRegressionCV blend over TabM seed outputs.
# Kept because this is ps6e6-one-vs-rest-tabm as-is.

def run_blend(target_id, drop_id, preds_tmp, test_tmp, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = target_id
    target_name = reverse_mapping[target_id]
    for fold, (tr_idx, val_idx) in tqdm(enumerate(skf.split(preds_tmp, target[target_name]))):
        X_train = preds_tmp.iloc[tr_idx].reset_index(drop=True)
        X_val = preds_tmp.iloc[val_idx].reset_index(drop=True)
        X_val_full = preds_tmp.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        # masking third class
        if drop_id:
            drop_name = reverse_mapping[drop_id]
            train_mask_2 = (train_target[drop_name] == 0)
            val_mask_2 = (val_target[drop_name] == 0)
        else:
            train_mask_2 = (train_target[target_name] >= 0)
            val_mask_2 = (val_target[target_name] >= 0)
        X_train = X_train[train_mask_2]
        train_target = train_target[train_mask_2]
        X_val = X_val[val_mask_2]
        val_target = val_target[val_mask_2]
    
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_tmp.copy()
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])
        
        model = LogisticRegressionCV(max_iter=1000, random_state=seed, Cs=10, scoring='roc_auc', cv=5)
    
        model.fit(
            X_train, train_target.iloc[:, i]
        )
    
        p_val = model.predict_proba(X_val)[:, 1]
        p_val_full = model.predict_proba(X_val_full)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        map_coef = pd.Series(model.coef_[0], index=model.feature_names_in_)
        print(map_coef.sort_values())
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_1 = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_1.to_parquet(f'oof_preds_{target_id}_blend_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    test_preds_1 = pd.DataFrame(p_test_x_final, columns=target.columns)
    test_preds_1['id'] = test_ids
    test_preds_1.to_parquet(f'test_preds_{target_id}_blend_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return oof_preds_1, test_preds_1

In [15]:
# ============================================================
# 6. Run one-vs-rest TabM and class-wise blend
# ============================================================

# QSO
preds_l_1 = []
test_l_1 = []
pred_col_1 = []
for model_type in MODEL_TYPES_1:
    for seed in SEED_LIST:
        run(model_type, 1, None, seed)
        preds_l_1.append(pd.read_parquet(f"oof_preds_1_{model_type}_{seed}.parquet")[["QSO"]].rename(columns={"QSO": f"my_base_{model_type}_seed_{seed}"}))
        test_l_1.append(pd.read_parquet(f"test_preds_1_{model_type}_{seed}.parquet")[["QSO"]].rename(columns={"QSO": f"my_base_{model_type}_seed_{seed}"}))
        pred_col_1.append(f"my_base_{model_type}_seed_{seed}")

if ADD_EXT:
    for cnt_1, oof_tmp in enumerate(add_oof):
        preds_l_1.append(oof_tmp.iloc[:, 1])
        pred_col_1.append(add_features[cnt_1])
    for test_tmp in add_test:
        test_l_1.append(test_tmp.iloc[:, 1])

preds_tmp_1 = pd.concat(preds_l_1, axis=1)
test_tmp_1 = pd.concat(test_l_1, axis=1)
preds_tmp_1 = to_logits(preds_tmp_1)
test_tmp_1 = to_logits(test_tmp_1)
preds_tmp_1.columns = pred_col_1
test_tmp_1.columns = pred_col_1
print("QSO blend input:", preds_tmp_1.shape, test_tmp_1.shape)

oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, preds_final_tmp = run_blend(1, None, preds_tmp_1, test_tmp_1, seed)
    oof_x.append(oof_preds_tmp.set_index("id"))
    P_x.append(preds_final_tmp.set_index("id"))
    train_ids = oof_preds_tmp[["id"]]
oof_preds_1 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_1.columns = target.columns
oof_preds_1["id"] = train_ids["id"].copy()
test_preds_1 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_1.columns = target.columns
test_preds_1["id"] = test_ids

# STAR
preds_l_2 = []
test_l_2 = []
pred_col_2 = []
for model_type in MODEL_TYPES_2:
    for seed in SEED_LIST:
        run(model_type, 2, None, seed)
        preds_l_2.append(pd.read_parquet(f"oof_preds_2_{model_type}_{seed}.parquet")[["STAR"]].rename(columns={"STAR": f"my_base_{model_type}_seed_{seed}"}))
        test_l_2.append(pd.read_parquet(f"test_preds_2_{model_type}_{seed}.parquet")[["STAR"]].rename(columns={"STAR": f"my_base_{model_type}_seed_{seed}"}))
        pred_col_2.append(f"my_base_{model_type}_seed_{seed}")

if ADD_EXT:
    for cnt_2, oof_tmp in enumerate(add_oof):
        preds_l_2.append(oof_tmp.iloc[:, 2])
        pred_col_2.append(add_features[cnt_2])
    for test_tmp in add_test:
        test_l_2.append(test_tmp.iloc[:, 2])

preds_tmp_2 = pd.concat(preds_l_2, axis=1)
test_tmp_2 = pd.concat(test_l_2, axis=1)
preds_tmp_2 = to_logits(preds_tmp_2)
test_tmp_2 = to_logits(test_tmp_2)
preds_tmp_2.columns = pred_col_2
test_tmp_2.columns = pred_col_2
print("STAR blend input:", preds_tmp_2.shape, test_tmp_2.shape)

oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, test_preds_tmp = run_blend(2, None, preds_tmp_2, test_tmp_2, seed)
    oof_x.append(oof_preds_tmp.set_index("id"))
    P_x.append(test_preds_tmp.set_index("id"))
    train_ids = oof_preds_tmp[["id"]]
oof_preds_2 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_2.columns = target.columns
oof_preds_2["id"] = train_ids["id"].copy()
test_preds_2 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_2.columns = target.columns
test_preds_2["id"] = test_ids

# GALAXY
preds_l_3 = []
test_l_3 = []
pred_col_3 = []
for model_type in MODEL_TYPES_3:
    for seed in SEED_LIST:
        run(model_type, 0, None, seed)
        preds_l_3.append(pd.read_parquet(f"oof_preds_0_{model_type}_{seed}.parquet")[["GALAXY"]].rename(columns={"GALAXY": f"my_base_{model_type}_seed_{seed}"}))
        test_l_3.append(pd.read_parquet(f"test_preds_0_{model_type}_{seed}.parquet")[["GALAXY"]].rename(columns={"GALAXY": f"my_base_{model_type}_seed_{seed}"}))
        pred_col_3.append(f"my_base_{model_type}_seed_{seed}")

if ADD_EXT:
    for cnt_3, oof_tmp in enumerate(add_oof):
        preds_l_3.append(oof_tmp.iloc[:, 0])
        pred_col_3.append(add_features[cnt_3])
    for test_tmp in add_test:
        test_l_3.append(test_tmp.iloc[:, 0])

preds_tmp_3 = pd.concat(preds_l_3, axis=1)
test_tmp_3 = pd.concat(test_l_3, axis=1)
preds_tmp_3 = to_logits(preds_tmp_3)
test_tmp_3 = to_logits(test_tmp_3)
preds_tmp_3.columns = pred_col_3
test_tmp_3.columns = pred_col_3
print("GALAXY blend input:", preds_tmp_3.shape, test_tmp_3.shape)

oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, test_preds_tmp = run_blend(0, None, preds_tmp_3, test_tmp_3, seed)
    oof_x.append(oof_preds_tmp.set_index("id"))
    P_x.append(test_preds_tmp.set_index("id"))
    train_ids = oof_preds_tmp[["id"]]
oof_preds_3 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_3.columns = target.columns
oof_preds_3["id"] = train_ids["id"].copy()
test_preds_3 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_3.columns = target.columns
test_preds_3["id"] = test_ids


QSO


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9978205448007998
QSO


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/rtdl_num_embeddings.py:311: UserWarning: The 7-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


0.9978645265493649
QSO


...

GALAXY
0.9958924002932665
my_base_t_seed_2809    0.291029
my_base_t_seed_60      0.346738
my_base_t_seed_0       0.393595
dtype: float64


4it [00:32,  7.98s/it]

GALAXY
0.9959513345848812
my_base_t_seed_2809    0.287332
my_base_t_seed_60      0.333823
my_base_t_seed_0       0.413048
dtype: float64


5it [00:39,  7.96s/it]


[np.float64(0.9958650943792706), np.float64(0.9955656908685039), np.float64(0.9958253282897715), np.float64(0.9958924002932665), np.float64(0.9959513345848812)]
0.9958199696831388


In [16]:
# ============================================================
# 7. Assemble 3-class probabilities and tune bias for submission
# ============================================================

oof_final = oof_preds_1[["id", "QSO"]].join(oof_preds_2[["STAR"]]).join(oof_preds_3[["GALAXY"]])
test_final = test_preds_1[["id", "QSO"]].join(test_preds_2[["STAR"]]).join(test_preds_3[["GALAXY"]])

oof_final_2 = oof_final[["GALAXY", "QSO", "STAR"]].copy()
test_final_2 = test_final[["GALAXY", "QSO", "STAR"]].copy()
oof_final_2 = oof_final_2.div(oof_final_2.sum(axis=1), axis=0)
test_final_2 = test_final_2.div(test_final_2.sum(axis=1), axis=0)

raw_oof_pred = np.argmax(oof_final_2.values, axis=1)
raw_oof_ba = balanced_accuracy_score(y, raw_oof_pred)
raw_recalls = recall_score(y, raw_oof_pred, average=None)

print(f"Raw OOF Balanced Accuracy: {raw_oof_ba:.9f}")
print("Raw recalls:", dict(zip(CLASS_NAMES, raw_recalls)))

best_bias, tuned_ba, opt_history = tune_bias(oof_final_2.values, y)
biased_oof_pred = public_preds(oof_final_2.values, best_bias)
biased_recalls = recall_score(y, biased_oof_pred, average=None)

print(f"Best bias [GALAXY, QSO, STAR]: {best_bias}")
print(f"Tuned OOF Balanced Accuracy: {tuned_ba:.9f} (+{tuned_ba - raw_oof_ba:.9f})")
print("Tuned recalls:", dict(zip(CLASS_NAMES, biased_recalls)))

test_raw_pred = np.argmax(test_final_2.values, axis=1)
test_biased_pred = public_preds(test_final_2.values, best_bias)

test_raw_labels = [reverse_mapping[p] for p in test_raw_pred]
test_biased_labels = [reverse_mapping[p] for p in test_biased_pred]

display(pd.DataFrame({
    "class": CLASS_NAMES,
    "raw_recall": raw_recalls,
    "tuned_recall": biased_recalls,
}))


Raw OOF Balanced Accuracy: 0.961010565
Raw recalls: {'GALAXY': np.float64(0.9798797287273497), 'QSO': np.float64(0.9659390659279684), 'STAR': np.float64(0.9372129007301387)}
Best bias [GALAXY, QSO, STAR]: [-1.42  -0.213  0.   ]
Tuned OOF Balanced Accuracy: 0.968616828 (+0.007606263)
Tuned recalls: {'GALAXY': np.float64(0.9574441029988344), 'QSO': np.float64(0.9771646619943146), 'STAR': np.float64(0.9712417194526377)}


,class,raw_recall,tuned_recall
0,GALAXY,0.979880,0.957444
1,QSO,0.965939,0.977165
2,STAR,0.937213,0.971242


In [17]:
# ============================================================
# 8. Save artifacts
# ============================================================

oof_proba = oof_final_2.values.astype("float32")
pred_proba = test_final_2.values.astype("float32")

np.save(OOF_NPY, oof_proba)
np.save(PRED_NPY, pred_proba)

oof_csv = pd.DataFrame(oof_proba, columns=CLASS_NAMES)
oof_csv.insert(0, ID, train_full["id"].values)
oof_csv[TARGET] = [reverse_mapping[v] for v in y]
oof_csv.to_csv(OOF_CSV, index=False)

pred_csv = pd.DataFrame(pred_proba, columns=CLASS_NAMES)
pred_csv.insert(0, ID, test_ids)
pred_csv.to_csv(PRED_CSV, index=False)

submission = pd.DataFrame({ID: test_ids, TARGET: test_biased_labels})
submission.to_csv(SUBMISSION_PATH, index=False)

submission_raw = pd.DataFrame({ID: test_ids, TARGET: test_raw_labels})
submission_raw.to_csv(SUBMISSION_RAW_PATH, index=False)

class_dist = pd.DataFrame({
    "class": CLASS_NAMES,
    "oof_raw_count": pd.Series(raw_oof_pred).value_counts().reindex([0, 1, 2], fill_value=0).values,
    "oof_tuned_count": pd.Series(biased_oof_pred).value_counts().reindex([0, 1, 2], fill_value=0).values,
    "test_raw_count": pd.Series(test_raw_pred).value_counts().reindex([0, 1, 2], fill_value=0).values,
    "test_tuned_count": pd.Series(test_biased_pred).value_counts().reindex([0, 1, 2], fill_value=0).values,
})
class_dist["test_tuned_ratio"] = class_dist["test_tuned_count"] / len(test_biased_pred)
class_dist.to_csv(CLASS_DISTRIBUTION_PATH, index=False)
display(class_dist)

feature_info = {
    "feature_family": "ps6e6_one_vs_rest_tabm_fe",
    "n_features": int(len(feature_cols)),
    "n_cat_features": int(len(cat_cols)),
    "cat_cols": list(cat_cols),
    "feature_cols": list(feature_cols),
    "te_columns": list(TE_columns),
    "seed_list": list(SEED_LIST),
    "model_types": {
        "qso": MODEL_TYPES_1,
        "star": MODEL_TYPES_2,
        "galaxy": MODEL_TYPES_3,
    },
    "add_external": bool(ADD_EXT),
}
with open(FEATURE_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_info, f, indent=2, ensure_ascii=False, default=str)

pd.DataFrame({"feature": feature_cols, "is_cat": [c in cat_cols for c in feature_cols]}).to_csv(FEATURE_COLUMNS_PATH, index=False)

cv_result = {
    "exp_id": EXP_ID,
    "competition": COMPETITION,
    "created_at": CREATED_AT,
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "model_family": "TabM",
    "model_type": "binary_one_vs_rest_tabm_with_classwise_logregcv_blend",
    "raw_cv": float(raw_oof_ba),
    "tuned_cv": float(tuned_ba),
    "best_bias": [float(x) for x in best_bias],
    "raw_recalls": {CLASS_NAMES[i]: float(raw_recalls[i]) for i in range(3)},
    "tuned_recalls": {CLASS_NAMES[i]: float(biased_recalls[i]) for i in range(3)},
    "n_splits": int(N_FOLDS),
    "seed": int(SEED),
    "seed_list": [int(s) for s in SEED_LIST],
    "use_original": False,
    "use_fe": True,
    "use_bias_search": True,
    "use_external_stacking_files": bool(ADD_EXT),
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": OOF_CSV.name,
        "pred_csv": PRED_CSV.name,
        "submission": SUBMISSION_PATH.name,
        "submission_raw": SUBMISSION_RAW_PATH.name,
        "feature_info": FEATURE_INFO_PATH.name,
        "feature_columns": FEATURE_COLUMNS_PATH.name,
        "class_distribution": CLASS_DISTRIBUTION_PATH.name,
    },
    "params_t": params_t,
}
with open(CV_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(cv_result, f, indent=2, ensure_ascii=False, default=str)

print("saved:", OOF_NPY)
print("saved:", PRED_NPY)
print("saved:", SUBMISSION_PATH)
print("saved:", CV_RESULT_PATH)


,class,oof_raw_count,oof_tuned_count,test_raw_count,test_tuned_count,test_tuned_ratio
0,GALAXY,377690,365004,161926,156332,0.631810
1,QSO,116826,120620,50083,51670,0.208823
2,STAR,82831,91723,35426,39433,0.159367


saved: /kaggle/working/oof_exp_20260609_032_ovr_tabm_as_is_proba.npy
saved: /kaggle/working/pred_exp_20260609_032_ovr_tabm_as_is_proba.npy
saved: /kaggle/working/submission_exp_20260609_032_ovr_tabm_as_is.csv
saved: /kaggle/working/cv_result_exp_20260609_032_ovr_tabm_as_is.json


In [18]:
# ============================================================
# 9. Update registry / memo
# ============================================================

registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "TabM",
    "feature_family": "ps6e6_one_vs_rest_tabm_fe",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(tuned_ba),
    "raw_cv_score": float(raw_oof_ba),
    "fold_std": np.nan,
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": True,
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
    "submission_path": str(SUBMISSION_PATH),
    "role": "tabm_ovr_family_material",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb_and_corr",
    "notes": (
        "ps6e6-one-vs-rest-tabm as-is. Runs TabM_D_Classifier for binary OVR "
        "GALAXY/QSO/STAR with SEED_LIST=[60,0,2809], class-wise LogisticRegressionCV over "
        "TabM seed outputs, and final bias tuning for submission. Saves exp_id-named "
        "OOF/pred probabilities for later 033 add032 blend/corr check."
    ),
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "ps6e6-one-vs-rest-tabm as-is",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "TabM_D_Classifier",
        "created_at": CREATED_AT,
    },
    "objective": {
        "summary": (
            "Run ps6e6-one-vs-rest-tabm as-is to obtain a TabM/NN-family OOF/pred material "
            "distinct from RealMLP, CatBoost, and XGB. Save exp_id-named OOF/pred probability "
            "files for the next add032 blend/corr check."
        ),
        "success_criteria": [
            "run TabM one-vs-rest training body",
            "use SEED_LIST=[60,0,2809]",
            "keep source notebook's class-wise LogisticRegressionCV over TabM seed outputs",
            "save raw OOF proba npy with exp_id in filename",
            "save raw test pred proba npy with exp_id in filename",
            "save bias-tuned submission",
            "save cv_result",
            "save feature_info",
            "update oof_registry",
        ],
    },
    "data": {
        "train_path": str(Path(PATH) / "train.csv"),
        "test_path": str(Path(PATH) / "test.csv"),
        "sample_submission_path": str(Path(PATH) / "sample_submission.csv"),
        "target_col": TARGET,
        "id_col": ID,
        "use_original": False,
        "use_external_stacking_files": bool(ADD_EXT),
    },
    "seed": {
        "seed": SEED,
        "seed_list": SEED_LIST,
        "random_seed": SEED,
        "numpy_seed": SEED,
        "torch_seed": SEED,
        "stratified_kfold_random_state_per_seed": SEED_LIST,
        "target_encoder_random_state_per_seed": SEED_LIST,
        "tabm_random_state": params_t.get("random_state"),
    },
    "features": {
        "feature_family": "ps6e6_one_vs_rest_tabm_fe",
        "feature_info": feature_info,
        "target_encoder_safety": (
            "TargetEncoder is fit inside each OVR fold on train fold combo features and binary labels, "
            "then transforms validation and test."
        ),
    },
    "model": {
        "family": "TabM",
        "type": "binary_one_vs_rest",
        "base_model": "TabM_D_Classifier",
        "classwise_blender": "LogisticRegressionCV",
        "params_t": params_t,
        "probability_postprocess": (
            "Build independent class probabilities from class-wise OVR outputs, "
            "then normalize each row to sum to 1."
        ),
        "bias_tuning": "Applied only to submission labels. OOF/pred npy store raw normalized probabilities.",
    },
    "cv": {
        "scheme": "StratifiedKFold OVR per class and seed",
        "n_splits": N_FOLDS,
        "shuffle": True,
        "raw_score": float(raw_oof_ba),
        "tuned_score": float(tuned_ba),
        "best_bias": [float(x) for x in best_bias],
        "raw_recalls": {CLASS_NAMES[i]: float(raw_recalls[i]) for i in range(3)},
        "tuned_recalls": {CLASS_NAMES[i]: float(biased_recalls[i]) for i in range(3)},
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": OOF_CSV.name,
        "pred_csv": PRED_CSV.name,
        "submission": SUBMISSION_PATH.name,
        "submission_raw": SUBMISSION_RAW_PATH.name,
        "cv_result": CV_RESULT_PATH.name,
        "feature_info": FEATURE_INFO_PATH.name,
        "feature_columns": FEATURE_COLUMNS_PATH.name,
        "class_distribution": CLASS_DISTRIBUTION_PATH.name,
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb_and_corr",
        "reason": (
            "This is a TabM/NN-family OVR material candidate. Adoption depends on CV, Public LB, "
            "and OOF correlation/blend behavior against 031/029/027/026/028/030/018."
        ),
        "next_action": [
            f"Submit {SUBMISSION_PATH.name}",
            "Record Public LB",
            "Add OOF/pred npy to npy-files dataset",
            "Run 033 add032 blend/corr check",
            "Compare against current best 031 and backup 029",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())


registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,raw_cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260609_032_ovr_tabm_as_is,TabM,ps6e6_one_vs_rest_tabm_fe,balanced_accuracy_score,0.968617,0.961011,NaN,NaN,False,True,True,/kaggle/working/oof_exp_20260609_032_ovr_tabm_...,/kaggle/working/pred_exp_20260609_032_ovr_tabm...,/kaggle/working/submission_exp_20260609_032_ov...,tabm_ovr_family_material,completed,pending_public_lb_and_corr,ps6e6-one-vs-rest-tabm as-is. Runs TabM_D_Clas...


In [19]:
# ============================================================
# 10. Optional cleanup of intermediate parquet files
# ============================================================

for p in Path(".").glob("*.parquet"):
    try:
        p.unlink()
    except Exception as e:
        print("skip cleanup:", p, e)
